# Downloading DWD weather data

This notebook downloads files linked from a DWD OpenData directory listing. The data is licensed [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.en) ([read more](https://www.dwd.de/EN/service/legal_notice/legal_notice_node.html)).

By default this targets the recent [10-minute air-temperature observations](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/air_temperature/recent/) and corresponding temperature data.

It will download files such as `rain_recent\10minutenwerte_nieder_00989_akt.zip` where `nieder` stands for precipitation and 00989 is the ID of a weather measurement station. You can find all stations and their corresponding meta data in the text file [zehn_min_rr_Beschreibung_Stationen.txt](rain_recent/zehn_min_rr_Beschreibung_Stationen.txt).

In [1]:
from __future__ import annotations

import argparse
import sys
from dataclasses import dataclass
from html.parser import HTMLParser
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import unquote, urljoin, urlparse
from urllib.request import Request, urlopen

# Temperature:
TEMP_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/air_temperature/now/"
TEMP_RECENT_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/air_temperature/recent/"

# Rain:
RAIN_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/precipitation/now/"
RAIN_RECENT_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/precipitation/recent/"


USER_AGENT = "dwd-linked-file-downloader/1.0 (+https://opendata.dwd.de)"


class LinkParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.hrefs: list[str] = []

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag.lower() != "a":
            return
        for name, value in attrs:
            if name.lower() == "href" and value:
                self.hrefs.append(value)


@dataclass(frozen=True)
class LinkedFile:
    url: str
    filename: str
    is_txt: bool


def fetch_bytes(url: str, timeout: int) -> bytes:
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout) as response:
        return response.read()


def content_length(url: str, timeout: int) -> int | None:
    request = Request(url, method="HEAD", headers={"User-Agent": USER_AGENT})
    try:
        with urlopen(request, timeout=timeout) as response:
            value = response.headers.get("Content-Length")
    except (HTTPError, URLError, TimeoutError):
        return None
    return int(value) if value and value.isdigit() else None


def linked_files(index_url: str, timeout: int) -> list[LinkedFile]:
    html = fetch_bytes(index_url, timeout).decode("utf-8", errors="replace")
    parser = LinkParser()
    parser.feed(html)

    files_by_url: dict[str, LinkedFile] = {}
    for href in parser.hrefs:
        absolute_url = urljoin(index_url, href)
        parsed = urlparse(absolute_url)
        filename = Path(unquote(parsed.path)).name

        if not filename or href.endswith("/") or filename in {".", ".."}:
            continue
        if parsed.scheme not in {"http", "https"}:
            continue

        files_by_url[absolute_url] = LinkedFile(
            url=absolute_url,
            filename=filename,
            is_txt=filename.casefold().endswith(".txt"),
        )

    return sorted(files_by_url.values(), key=lambda item: (not item.is_txt, item.filename.casefold()))


def download_file(file: LinkedFile, out_dir: Path, overwrite: bool, timeout: int) -> str:
    target = out_dir / file.filename
    expected_size = None if overwrite else content_length(file.url, timeout)

    if target.exists() and not overwrite:
        actual_size = target.stat().st_size
        if expected_size is None or actual_size == expected_size:
            return f"skip  {target} already exists"

    request = Request(file.url, headers={"User-Agent": USER_AGENT})
    tmp_target = target.with_suffix(target.suffix + ".part")
    with urlopen(request, timeout=timeout) as response, tmp_target.open("wb") as output:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)

    tmp_target.replace(target)
    return f"saved {target}"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Inspect a DWD directory listing and download all directly linked files, .txt files first."
    )
    parser.add_argument("url", nargs="?", default=DEFAULT_URL, help=f"Directory URL. Default: {DEFAULT_URL}")
    parser.add_argument("-o", "--out-dir", default="dwd_air_temperature_now", help="Download directory.")
    parser.add_argument("--overwrite", action="store_true", help="Redownload files even if they already exist.")
    parser.add_argument("--timeout", type=int, default=60, help="Network timeout in seconds.")
    return parser.parse_args()


def load_weather_data(url, folder):
    out_dir = Path("./" + folder)
    out_dir.mkdir(parents=True, exist_ok=True)
    args_url = url
    args_timeout= 30
    args_overwrite = True

    try:
        files = linked_files(args_url, args_timeout)
    except (HTTPError, URLError, TimeoutError) as exc:
        print(f"Could not inspect {args_url}: {exc}", file=sys.stderr)
        return 1

    if not files:
        print(f"No linked files found at {args_url}", file=sys.stderr)
        return 1

    txt_count = sum(file.is_txt for file in files)
    print(f"Found {len(files)} linked files ({txt_count} .txt). Downloading .txt files first.")

    failures = 0
    for file in files:
        try:
            print(download_file(file, out_dir, args_overwrite, args_timeout))
        except (HTTPError, URLError, TimeoutError, OSError) as exc:
            failures += 1
            print(f"fail  {file.url}: {exc}", file=sys.stderr)

    if failures:
        print(f"Completed with {failures} failed download(s).", file=sys.stderr)
        return 1

    print(f"Done. Files are in {out_dir.resolve()}")
    return 0

#load_weather_data(RAIN_URL, "rain")
#load_weather_data(TEMP_URL, "temp")
load_weather_data(RAIN_RECENT_URL, "rain_recent")
load_weather_data(TEMP_RECENT_URL, "temp_recent")

Found 1400 linked files (1 .txt). Downloading .txt files first.
saved rain_recent\zehn_min_rr_Beschreibung_Stationen.txt
saved rain_recent\10minutenwerte_nieder_00020_akt.zip
saved rain_recent\10minutenwerte_nieder_00029_akt.zip
saved rain_recent\10minutenwerte_nieder_00044_akt.zip
saved rain_recent\10minutenwerte_nieder_00046_akt.zip
saved rain_recent\10minutenwerte_nieder_00053_akt.zip
saved rain_recent\10minutenwerte_nieder_00073_akt.zip
saved rain_recent\10minutenwerte_nieder_00078_akt.zip
saved rain_recent\10minutenwerte_nieder_00087_akt.zip
saved rain_recent\10minutenwerte_nieder_00091_akt.zip
saved rain_recent\10minutenwerte_nieder_00096_akt.zip
saved rain_recent\10minutenwerte_nieder_00103_akt.zip
saved rain_recent\10minutenwerte_nieder_00118_akt.zip
saved rain_recent\10minutenwerte_nieder_00124_akt.zip
saved rain_recent\10minutenwerte_nieder_00130_akt.zip
saved rain_recent\10minutenwerte_nieder_00131_akt.zip
saved rain_recent\10minutenwerte_nieder_00141_akt.zip
saved rain_rece

0